# Overview

- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28×28 grayscale image
  + Output: one of 10 digit classes (0–9)
- Converting the Keras CNN + MLP example to PyTorch
- Training and evaluating a CNN + MLP with shared training utilities


# Import libraries

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset, TensorDataset

project = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file() and (path / "1. MLP").is_dir()
)
work_dir = project / "2. CNN + MLP"
utility_dir = project / "4. Training Utilities"
sys.path.insert(0, str(utility_dir))
from shared_training import fit, run_epoch, seed_everything

SEED = 42
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MNIST_ARCHIVE = work_dir / "mnist.npz"
RESULTS_DIR = work_dir / "results"
RUN_DIR = RESULTS_DIR / time.strftime("%Y%m%d_%H%M%S")
LOG_DIR = work_dir / "runs" / RUN_DIR.name
seed_everything(SEED)
print("Device:", DEVICE)
print("Run directory:", RUN_DIR)

# (1) Data

Load the local `mnist.npz` archive next to this notebook. It contains the same predefined 60,000-image training split and 10,000-image test split used by the Keras example. Images are converted to NCHW `float32` tensors and scaled to [0, 1], matching the Keras division by 255; integer labels are used with `CrossEntropyLoss`.

To match the original `model.fit(..., validation_data=(x_test, y_test))` call, the test split is also used for validation and checkpoint selection, then evaluated again afterward. The final metrics are therefore not from an independent held-out set.

In [ ]:
if not MNIST_ARCHIVE.is_file():
    raise FileNotFoundError(f"MNIST archive not found: {MNIST_ARCHIVE}")

with np.load(MNIST_ARCHIVE) as archive:
    train_images = archive["x_train"]
    train_labels = archive["y_train"]
    test_images = archive["x_test"]
    test_labels = archive["y_test"]

if train_images.shape != (60000, 28, 28) or test_images.shape != (10000, 28, 28):
    raise ValueError("Expected MNIST image arrays with shapes (60000, 28, 28) and (10000, 28, 28).")
if train_labels.shape != (60000,) or test_labels.shape != (10000,):
    raise ValueError("Expected MNIST label arrays with shapes (60000,) and (10000,).")

x_train = torch.from_numpy(
    np.ascontiguousarray(train_images[:, None], dtype=np.float32) / 255.0
)
y_train = torch.from_numpy(np.ascontiguousarray(train_labels, dtype=np.int64))
x_test = torch.from_numpy(
    np.ascontiguousarray(test_images[:, None], dtype=np.float32) / 255.0
)
y_test = torch.from_numpy(np.ascontiguousarray(test_labels, dtype=np.int64))

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)
valid_dataset = test_dataset

loader_generator = torch.Generator().manual_seed(SEED)
pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=loader_generator, pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory
)
valid_loader = test_loader
images, labels = next(iter(train_loader))
print("Training/validation/test samples:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("MNIST archive:", MNIST_ARCHIVE)
print("Image batch shape:", tuple(images.shape), "Label shape:", tuple(labels.shape))
print("Pixel range:", images.min().item(), images.max().item())

# (2) Model

The Keras layers in the example map to PyTorch as follows:

| Keras | PyTorch |
|---|---|
| `Conv2D`, NHWC layout | `nn.Conv2d`, NCHW layout |
| `MaxPooling2D`, `Flatten` | `nn.MaxPool2d`, `nn.Flatten` |
| `Dense`, `Dropout`, ReLU | `nn.Linear`, `nn.Dropout`, `F.relu` |
| Softmax + categorical cross-entropy | logits + `nn.CrossEntropyLoss` |

The model preserves the three convolution blocks (64/128/256 channels), each followed by 2×2 max pooling and 0.2 dropout, then an MLP with hidden dimensions 512 → 512 and 10 output logits. Three pooling operations reduce the image size from 28 → 14 → 7 → 3, so the first linear layer receives 256×3×3 features.

In [ ]:
class MNISTCNNMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout2 = nn.Dropout(0.2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout3 = nn.Dropout(0.2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.dropout_fc1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 512)
        self.dropout_fc2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(512, num_classes)

    def conv_features(self, images):
        x = self.dropout1(self.pool1(F.relu(self.conv1(images))))
        x = self.dropout2(self.pool2(F.relu(self.conv2(x))))
        return self.pool3(F.relu(self.conv3(x)))

    def penultimate_features(self, images):
        x = self.dropout3(self.conv_features(images))
        x = self.dropout_fc1(F.relu(self.fc1(self.flatten(x))))
        return F.relu(self.fc2(x))

    def forward(self, images):
        x = self.dropout_fc2(self.penultimate_features(images))
        return self.fc3(x)


model = MNISTCNNMLP().to(DEVICE)
with torch.inference_mode():
    sample_logits = model(torch.zeros(2, 1, 28, 28, device=DEVICE))
assert sample_logits.shape == (2, 10)
print(model)
print("Output logits:", tuple(sample_logits.shape))
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))

# (3) Training

The shared `shared_training.py` module provides the train/evaluation loop, best-validation-loss checkpointing, early stopping, `ReduceLROnPlateau`, reproducible seeding, and TensorBoard scalar logging. This notebook keeps the example's SGD learning rate (0.01), batch size (128), and maximum of 30 epochs. Early stopping uses patience 7 and the scheduler uses patience 2 so it can reduce the learning rate before training stops. Each run is saved in a new timestamped directory.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.1, patience=2, min_lr=0.0001
)
checkpoint_path = RUN_DIR / "best.pth"
RUN_TRAINING = True

if RUN_TRAINING:
    from torch.utils.tensorboard import SummaryWriter

    RUN_DIR.mkdir(parents=True, exist_ok=False)
    with SummaryWriter(log_dir=str(LOG_DIR)) as writer:
        training_result = fit(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=DEVICE,
            epochs=EPOCHS,
            checkpoint_path=checkpoint_path,
            patience=7,
            min_delta=0.0,
            scheduler=scheduler,
            writer=writer,
        )
    history = pd.DataFrame(training_result["history"])
    history.to_csv(RUN_DIR / "history.csv", index=False)
    summary = {key: value for key, value in training_result.items() if key != "history"}
    (RUN_DIR / "training.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    display(history.tail())
    print("Best epoch:", summary["best_epoch"])
    print("Best validation loss:", summary["best_val_loss"])
    print("Best validation accuracy:", summary["best_val_accuracy"])
    print("Training time (seconds):", round(summary["training_seconds"], 2))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["epoch"], history["train_loss"], label="Train")
    axes[0].plot(history["epoch"], history["val_loss"], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].plot(history["epoch"], history["train_accuracy"], label="Train")
    axes[1].plot(history["epoch"], history["val_accuracy"], label="Validation")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy")
    for axis in axes:
        axis.legend()
    fig.tight_layout()
    plt.show()

# (4) Evaluation

The Keras example uses its test split for validation and evaluates the same split after training. This notebook preserves that behavior, so the final metrics are not from an independent held-out set. Metrics are saved to `evaluation.json` in the run directory.

In [ ]:
if not checkpoint_path.is_file():
    raise FileNotFoundError("Run the training cell to create the checkpoint: " + str(checkpoint_path))
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
test_metrics = run_epoch(model, test_loader, criterion, DEVICE)
evaluation = {
    "test_loss": test_metrics["loss"],
    "test_accuracy": test_metrics["accuracy"],
    "test_samples": len(test_dataset),
    "best_epoch": training_result["best_epoch"],
}
(RUN_DIR / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
print(f"Test loss: {evaluation['test_loss']:.4f}")
print(f"Test accuracy: {evaluation['test_accuracy']:.2%}")

In [ ]:
random_test_index = int(np.random.default_rng(SEED).integers(len(test_dataset)))
test_image, test_label = test_dataset[random_test_index]
model.eval()
with torch.inference_mode():
    predicted_label = model(test_image.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
plt.figure(figsize=(4, 4))
plt.imshow(test_image.squeeze(0), cmap="gray")
plt.title(f"True label: {test_label} | Predicted: {predicted_label}")
plt.axis("off")
plt.show()

# (5) Miscellaneous

After training, run this cell to visualize the first convolution's learned filters and three PCA views (raw pixels, `fc2` features, and global-average-pooled `pool3` features). PCA uses a reproducible sample of up to 3,000 training images to keep computation manageable. It is enabled by default; set `RUN_PCA = False` to skip the PCA plots.

In [ ]:
weights = model.conv1.weight.detach().cpu().numpy()[:, 0]
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
limit = max(float(np.abs(weights).max()), 1e-12)
for index, axis in enumerate(axes.flat):
    axis.imshow(weights[index], cmap="seismic", vmin=-limit, vmax=limit)
    axis.set_title(str(index + 1), fontsize=7)
    axis.axis("off")
fig.suptitle("First convolution: 64 learned filters")
fig.tight_layout()
plt.show()

RUN_PCA = True
if RUN_PCA:
    sample_count = min(3000, len(train_dataset))
    selected = np.random.default_rng(SEED).choice(len(train_dataset), sample_count, replace=False)
    feature_loader = DataLoader(
        Subset(train_dataset, selected.tolist()), batch_size=BATCH_SIZE, shuffle=False
    )
    raw_features, fc_features, gap_features, pca_labels = [], [], [], []
    model.eval()
    with torch.inference_mode():
        for batch_images, batch_labels in feature_loader:
            batch_images = batch_images.to(DEVICE)
            conv = model.conv_features(batch_images)
            raw_features.append(batch_images.flatten(1).cpu().numpy())
            fc_features.append(model.penultimate_features(batch_images).cpu().numpy())
            gap_features.append(conv.mean(dim=(2, 3)).cpu().numpy())
            pca_labels.append(batch_labels.numpy())
    feature_sets = [
        ("Input pixels", np.concatenate(raw_features)),
        ("FC2 features", np.concatenate(fc_features)),
        ("Pool3 global-average features", np.concatenate(gap_features)),
    ]
    pca_labels = np.concatenate(pca_labels)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for axis, (title, features) in zip(axes, feature_sets):
        embedding = PCA(n_components=2, random_state=SEED).fit_transform(features)
        points = axis.scatter(
            embedding[:, 0], embedding[:, 1], c=pca_labels,
            cmap="tab10", s=5, alpha=0.65,
        )
        axis.set_title(title)
        axis.set_xlabel("PC1")
        axis.set_ylabel("PC2")
    fig.colorbar(points, ax=axes, ticks=range(10), label="Digit label")
    fig.tight_layout()
    plt.show()